# 19 · M1.4a · Repaired CRT samplers: Type I, power and audit validation

**Status:** executes the plan declared in [docs/m14_sampler_experiment_plan.md](../docs/m14_sampler_experiment_plan.md) (D-081). Every criterion, threshold and dataset count used below was fixed there before this notebook was written. This is the only design iteration (plan §1, §9).

**What runs here** (no real label is related to the real images anywhere in this notebook):
1. A label-free dependence analysis of the real CPER visual residuals. It fixes the gap between held-out blocks (plan §5) and checks the exchangeability assumption of candidate R (plan §6).
2. Two label mechanisms, LR and LP, fitted to the real labels given the context only (plan §3), with a label-only audit.
3. **Type I** of two candidate samplers (P, R) and five deliberately misspecified ones (M-a to M-e) in five null worlds (plan §4):
   - W0R and W0P: the real CPER images with synthetic labels;
   - W1: regime switching;
   - W2: GARCH-type volatility;
   - W3: long memory.
4. **Power** of the eligible candidates on the real images at planted 0.03 and 0.05 nats (plan §7).
5. **Validation of the held-out audit** on W1–W3: its threshold τ_A, its sensitivity to samplers that inflate Type I, and its false alarms (plan §5).
6. The decisions fixed for notebook 20 (plan §8).

Results are cached per section under `data/interim/m14/` so an interrupted run resumes; the cache never changes a criterion.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import beta

from visual_hygrometry import data, features, inference, samplers, scoring, splits, suite, worlds

warnings.filterwarnings('ignore', message='Mean of empty slice')
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 80, 'figure.figsize': (11, 3.2), 'axes.grid': True, 'grid.alpha': .3})
pd.set_option('display.precision', 3)
OFF, N_JOBS, N_PERM = -7, 18, 39
CACHE = data.ROOT / 'data' / 'interim' / 'm14'
CACHE.mkdir(parents=True, exist_ok=True)
par = lambda: Parallel(n_jobs=N_JOBS, max_nbytes=None)

def cached(name, f):
    """Load a section's results if a previous run stored them, else compute and store them."""
    p = CACHE / f'{name}.pkl'
    if p.exists(): return pd.read_pickle(p)
    t = time.time()
    out = f()
    pd.to_pickle(out, p)
    print(f'{name}: {(time.time() - t) / 60:.1f} min')
    return out

U = pd.read_csv(data.ROOT / 'data' / 'interim' / 'm13_CPER.csv', parse_dates=['utc'])  # notebook 17's usable rows
VIS = [n for n in features.NAMES if not ((U.groupby('year').qc_sky_clip.median() > .2).any() and n.startswith('vis_sky'))]
loc = (U.utc + pd.Timedelta(hours=OFF)).dt.tz_localize(None)
R0 = pd.DataFrame({'year': U.year - 2020, 'doy': loc.dt.day_of_year, 'hour_local': loc.dt.hour + loc.dt.minute / 60,
                   'day': (loc.dt.floor('D') - pd.Timestamp('2020-01-01')).dt.days, 'temp_cond_c': U.t_asp, 'd_ref_k': U.D_h})
R0 = pd.concat([R0, U[VIS], data.fourier(R0.hour_local, 24, 2, 'hour'), data.solar(R0.doy, R0.hour_local, suite.LAT, suite.LON, OFF)], axis=1)
yr, day, hour, temp, doy = (R0[c].to_numpy() for c in ('year', 'day', 'hour_local', 'temp_cond_c', 'doy'))
wk = yr * 14 + (doy - suite.DOY0) // 7
CRAW = np.c_[R0[['hour_local', 'temp_cond_c', 'doy', 'solar_elev_deg', 'solar_dayfrac']].to_numpy(), yr]  # as in suite.pipeline
V0, D0 = R0[VIS].to_numpy(), R0.d_ref_k.to_numpy()
sp = splits.kfold(yr)
DS = worlds.split_levels(R0, sp, suite.KNOTS, suite.DOY_K, suite.DOY_F, [], hour_season=True, solar=True)[0]['S']  # context fixed at S (plan §4)
dfl, lv = worlds.add_context_levels(R0, suite.KNOTS, suite.DOY_K, suite.DOY_F, hour_season=True, solar=True)
CN, KEEP = dfl[lv['S4']].to_numpy(), scoring.support_mask(CRAW[:, :2], sp)
F0 = inference.project(CRAW, V0, wk)  # cross-fitted E[V | C] of the real features (no label)
E0 = V0 - F0
SD, CORR = E0.std(0), np.corrcoef(E0.T)
print(f'{len(R0)} rows, {len(np.unique(day))} days, 4 seasons; visual features: {VIS}')

4817 rows, 364 days, 4 seasons; visual features: ['vis_sky_lum', 'vis_sky_br', 'vis_far_contrast', 'vis_near_contrast', 'vis_contrast_ratio', 'vis_far_sat', 'vis_near_gcc', 'vis_near_dark', 'vis_far_sharp']


## 1 · Label-free dependence analysis and the exchangeability checks for candidate R

The gap g between a held-out block and its fitting days is the smallest lag at which the median |ACF| of day-mean residuals (over features and seasons) falls below 0.1, bounded to [2, 7] days (plan §5).

In [2]:
def acf_days(E, k):
    out = []
    for s in range(4):
        r = yr == s
        _, inv = np.unique(day[r], return_inverse=True)
        u = np.array([E[r][inv == j].mean(0) for j in range(inv.max() + 1)])
        out += [np.corrcoef(u[k:, j], u[:-k, j])[0, 1] for j in range(E.shape[1])]
    return np.median(np.abs(out))

GAP = samplers.acf_gap(E0, day, yr)
print('median |ACF| of day means by lag:', {k: round(acf_days(E0, k), 3) for k in range(1, 11)}, f'-> gap g = {GAP} days')

median |ACF| of day means by lag: {1: np.float64(0.375), 2: np.float64(0.188), 3: np.float64(0.109), 4: np.float64(0.083), 5: np.float64(0.095), 6: np.float64(0.077), 7: np.float64(0.09), 8: np.float64(0.09), 9: np.float64(0.147), 10: np.float64(0.107)} -> gap g = 4 days


**Exchangeability of within-day deviation days given (season, daily-max-T tercile)** (plan §6). The checks compare within-day deviation statistics (SD and lag 1 of the day-demeaned residuals, median over features) between the first and second half of each season, and between the cooler and warmer half of days *within* a tercile.

In [3]:
dm = pd.DataFrame(E0).groupby(day).transform('mean').to_numpy()
W_ = E0 - dm
dd = pd.DataFrame({'season': yr, 'day': day, 'doy': doy, 'tmax': pd.Series(temp).groupby(day).transform('max').to_numpy()}).drop_duplicates('day').set_index('day')
dd['tercile'] = dd.groupby('season').tmax.transform(lambda x: pd.qcut(x, 3, labels=False))
dd['late'] = dd.groupby('season').doy.transform(lambda x: x > x.median())
dd['warm_in_tercile'] = dd.groupby(['season', 'tercile']).tmax.transform(lambda x: x > x.median())

def day_stats(rows):
    w, o = W_[rows], np.lexsort((hour[rows], day[rows]))
    w, same = w[o], day[rows][o][1:] == day[rows][o][:-1]
    lag1 = [np.corrcoef(w[1:, j][same], w[:-1, j][same])[0, 1] for j in range(w.shape[1])]
    return np.median(w.std(0)), np.median(lag1)

ex = []
for s in range(4):
    for split in ('late', 'warm_in_tercile'):
        a, b = (np.isin(day, dd.index[(dd.season == s) & (dd[split] == v)]) for v in (False, True))
        (sa, la), (sb, lb) = day_stats(a), day_stats(b)
        ex.append(dict(season=2020 + s, split=split, sd_ratio=sb / sa, lag1_a=la, lag1_b=lb))
pools = dd.groupby(['season', 'tercile']).size().unstack().rename(index=lambda s: 2020 + s)
print('donor pool sizes (days) per season x daily-max-T tercile:'); display(pools)
pd.DataFrame(ex)

donor pool sizes (days) per season x daily-max-T tercile:


tercile,0,1,2
season,,,
2020,31,30,31
2021,30,29,29
2022,31,30,31
2023,31,30,31


,season,split,sd_ratio,lag1_a,lag1_b
0,2020,late,1.058,0.186,0.238
1,2020,warm_in_tercile,0.935,0.230,0.200
2,2021,late,0.968,0.307,0.208
3,2021,warm_in_tercile,1.033,0.314,0.295
4,2022,late,1.099,0.157,0.177
5,2022,warm_in_tercile,0.989,0.192,0.143
6,2023,late,1.202,0.164,0.135
7,2023,warm_in_tercile,0.965,0.137,0.186


## 2 · Label mechanisms (labels given C only; V is never read)

M̂(C) is the out-of-fold mean of the context-only model with context S, leave one season out. ε = D − M̂(C). LR and LP are candidate R's and P's residual structures applied to ε. The audit compares their simulations with the real ε, and is reported, not used as a gate (plan §3).

In [4]:
M0 = scoring.crossfit(DS, D0, sp, wk, robust=True)[0]
EPS = (D0 - M0)[:, None]
LAB = {k: samplers.fit_process(EPS, day, hour, yr, hour, temp, kind=k) for k in ('R', 'P')}
lab_stats = lambda e: inference.residual_stats(e, day, hour, yr, hour, temp)
lab_env = {k: inference.envelope(lab_stats(EPS), [lab_stats(L.simulate(np.random.default_rng(9_000 + b), day, hour, yr, hour, temp)) for b in range(200)])
           for k, L in LAB.items()}
pd.DataFrame({f'L{k}': e.groupby(level=0).outside.mean() for k, e in lab_env.items()}).rename_axis('statistic (share of seasons outside the 95 % band)')

,LR,LP
statistic (share of seasons outside the 95 % band),,
day_lag1,0.00,0.50
day_lag7,0.25,0.25
kurt,0.00,0.50
sd_hour,0.00,0.00
sd_temp,0.00,0.50
skew,0.00,0.25
tail3,0.00,0.00
within_lag1,0.00,0.00


## 3 · Type I in five null worlds

Each dataset draws D* = M̂(C) + ε* with ε* from the world's label mechanism, independent of V given C. The CRT uses the validated statistic with context S, 39 draws, and rejects at p ≤ 0.05 (exact level 0.05).
- **Dataset counts:** 400 datasets per (candidate, world), and 200 per (misspecified sampler, world).
- **Sampler fitting:** in W0 each sampler is fitted once on the real V; in W1–W3 it is refitted on every dataset.
- **Classification** (one-sided 95 % Clopper–Pearson bounds, tolerance 0.10): acceptable if the upper bound ≤ 0.10; demonstrated inflation if the lower bound > 0.10; otherwise insufficient precision.

In [5]:
SAMPLERS = {'P': samplers.factory('P'), 'R': samplers.factory('R'), 'M-a': inference.conditional_sampler,
            'M-b': samplers.factory('P', phi_w=0.0), 'M-c': samplers.factory('P', gaussian=True, scale=False),
            'M-d': samplers.factory('P', independent=True), 'M-e': samplers.factory('P', persist=1.5)}
WORLDS = {'W0R': ('real', 'R'), 'W0P': ('real', 'P'), 'W1 regime': ('regime', 'P'), 'W2 volatility': ('garch', 'P'), 'W3 long memory': ('longmem', 'P')}
N_DATA = {k: 400 if k in ('P', 'R') else 200 for k in SAMPLERS}
DRAW0 = {k: f(V0, CRAW, day, hour, strata=yr, groups=wk) for k, f in SAMPLERS.items()}  # W0: V fixed, so each sampler is fitted once
Z1 = np.linalg.svd((E0 - E0.mean(0)) / E0.std(0), full_matrices=False)[0][:, 0]
Z1 = (Z1 - Z1.mean()) / Z1.std()
SIG_EPS = EPS.std()
seed_of = lambda s, w, i, c=0: 10_000_000 + list(SAMPLERS).index(s) * 1_000_000 + list(WORLDS).index(w) * 100_000 + c * 10_000 + i

def dataset(world, rng, c=0.0):
    """(V, D*, pre-fitted draw or None) of one dataset; c > 0 plants b·z with b = σ_ε √(e^{2c} − 1) (plan §3)."""
    kind, lab = WORLDS[world]
    V = V0 if kind == 'real' else F0 + samplers.world_residuals(kind, day, hour, yr, hour, temp, SD, CORR, rng)
    y = M0 + LAB[lab].simulate(rng, day, hour, yr, hour, temp)[:, 0] + SIG_EPS * np.sqrt(np.expm1(2 * c)) * Z1
    return V, y

def chunk(s, w, idx, c=0.0, ci=0):
    out = []
    for i in idx:
        seed = seed_of(s, w, i, ci)
        V, y = dataset(w, np.random.default_rng(seed), c)
        obs, p = samplers.crt(DS, V, y, sp, wk, hour, KEEP, CN, SAMPLERS[s], CRAW, day, yr, n_perm=N_PERM, seed=seed,
                              draw=DRAW0[s] if WORLDS[w][0] == 'real' else None)
        out.append(dict(sampler=s, world=w, i=i, c=c, obs=obs, p=p))
    return out

def run(jobs, size=10):
    tasks = [(s, w, list(range(a, min(a + size, n))), c, ci) for s, w, n, c, ci in jobs for a in range(0, n, size)]
    return pd.DataFrame([r for part in par()(delayed(chunk)(*t) for t in tasks) for r in part])

T1 = cached('typeI', lambda: run([(s, w, N_DATA[s], 0.0, 0) for s in SAMPLERS for w in WORLDS]))

def cp(x, n):
    """One-sided 95 % Clopper–Pearson bounds of a rejection rate."""
    return (beta.ppf(.05, x, n - x + 1) if x > 0 else 0.0), (beta.ppf(.95, x + 1, n - x) if x < n else 1.0)

def classify(g):
    x, n = int((g.p <= .05).sum()), len(g)
    lo, hi = cp(x, n)
    return pd.Series(dict(n=n, rejections=x, rate=x / n, lo=lo, hi=hi,
                          outcome='acceptable' if hi <= .10 else 'demonstrated inflation' if lo > .10 else 'insufficient precision'))

TAB = T1.groupby(['sampler', 'world']).apply(classify)
display(TAB.rate.unstack().loc[list(SAMPLERS), list(WORLDS)].round(3))
TAB.outcome.unstack().loc[list(SAMPLERS), list(WORLDS)]

typeI: 535.8 min


world,W0R,W0P,W1 regime,W2 volatility,W3 long memory
sampler,,,,,
P,0.080,0.035,0.072,0.060,0.075
R,0.072,0.050,0.085,0.075,0.112
M-a,0.040,0.030,0.055,0.075,0.045
M-b,0.055,0.035,0.090,0.090,0.105
M-c,0.065,0.040,0.075,0.060,0.090
M-d,0.115,0.080,0.085,0.085,0.080
M-e,0.050,0.030,0.040,0.025,0.065


world,W0R,W0P,W1 regime,W2 volatility,W3 long memory
sampler,,,,,
P,insufficient precision,acceptable,acceptable,acceptable,insufficient precision
R,acceptable,acceptable,insufficient precision,insufficient precision,insufficient precision
M-a,acceptable,acceptable,acceptable,insufficient precision,acceptable
M-b,acceptable,acceptable,insufficient precision,insufficient precision,insufficient precision
M-c,insufficient precision,acceptable,insufficient precision,acceptable,insufficient precision
M-d,insufficient precision,insufficient precision,insufficient precision,insufficient precision,insufficient precision
M-e,acceptable,acceptable,acceptable,acceptable,insufficient precision


In [6]:
OUT = TAB.outcome.unstack()
ELIGIBLE = [k for k in ('P', 'R') if (OUT.loc[k] == 'acceptable').all()]
CLASS = {k: 'invalid' if (OUT.loc[k] == 'demonstrated inflation').any() else 'acceptable' if (OUT.loc[k] == 'acceptable').all() else 'unclassified'
         for k in SAMPLERS if k.startswith('M-')}
print('eligible candidates (acceptable in all five null worlds):', ELIGIBLE or 'none')
print('misspecified samplers:', CLASS)

eligible candidates (acceptable in all five null worlds): none
misspecified samplers: {'M-a': 'unclassified', 'M-b': 'unclassified', 'M-c': 'unclassified', 'M-d': 'unclassified', 'M-e': 'unclassified'}


## 4 · Power on the real images (eligible candidates)

A planted signal c of 0.03 and 0.05 nats (a Gaussian approximation) is added along the first principal component of the real visual residuals. Three quantities are kept apart:
- the planted c;
- the true conditional mutual information, which differs because ε* is heavy-tailed and heteroscedastic;
- the observed held-out gain G, reported as the mean observed contrast.

There are 200 datasets per (candidate, world, c). Intervals are two-sided 95 % Clopper–Pearson.

In [7]:
POW = cached('power', lambda: run([(s, w, 200, c, ci) for s in ELIGIBLE for w in ('W0R', 'W0P') for ci, c in ((1, .03), (2, .05))])) if ELIGIBLE else pd.DataFrame()
if len(POW):
    def cp2(g):
        x, n = int((g.p <= .05).sum()), len(g)
        return pd.Series(dict(power=x / n, lo=beta.ppf(.025, x, n - x + 1) if x else 0.0, hi=beta.ppf(.975, x + 1, n - x) if x < n else 1.0, mean_G=g.obs.mean()))
    pw = POW.groupby(['sampler', 'world', 'c']).apply(cp2)
    display(pw)
else: print('no eligible candidate: power is not estimated (plan §7)')

no eligible candidate: power is not estimated (plan §7)


## 5 · Validation of the held-out audit (W1–W3)

There are 40 visual realisations per (sampler, world): 20 calibrate the threshold and 20 evaluate the audit.
- **Per realisation:** f̂ is the cross-fitted fit on all rows. The residual process is fitted on the training blocks of each season (4 contiguous blocks, gap g) and simulated 100 times for the held-out block.
- **Decision statistic:** A is the largest share of cells outside the 95 % band over the families H1–H6.
- **Threshold:** τ_A = max(0.10, the 90th percentile of A over the calibration realisations of every pair classed acceptable in §3).
- **Validated if:** the mean flag rate over invalid pairs is ≥ 0.80, and the flag rate of every acceptable pair is ≤ 0.20 (on the evaluation realisations).

In [8]:
KIND = {'P': ('P', {}), 'R': ('R', {}), 'M-a': ('nb17', {}), 'M-b': ('P', {'phi_w': 0.0}), 'M-c': ('P', {'gaussian': True, 'scale': False}),
        'M-d': ('P', {'independent': True}), 'M-e': ('P', {'persist': 1.5})}

def audit_one(s, w, i):
    rng = np.random.default_rng(50_000_000 + list(SAMPLERS).index(s) * 1_000_000 + list(WORLDS).index(w) * 100_000 + i)
    V = F0 + samplers.world_residuals(WORLDS[w][0], day, hour, yr, hour, temp, SD, CORR, rng)
    E = V - inference.project(CRAW, V, wk)
    kind, opts = KIND[s]
    share, A = samplers.audit_score(samplers.holdout_audit(E, day, hour, yr, hour, temp, kind, GAP, n_sim=100, seed=i, **opts))
    return dict(sampler=s, world=w, i=i, A=A) | {f'share_{k}': v for k, v in share.items()}

AUD = cached('audit', lambda: pd.DataFrame(par()(delayed(audit_one)(s, w, i) for s in SAMPLERS for w in list(WORLDS)[2:] for i in range(40))))
AUD['split'] = np.where(AUD.i < 20, 'calibration', 'evaluation')
pair = AUD.merge(TAB.outcome.rename('typeI').reset_index(), on=['sampler', 'world'])
pair['cls'] = np.where(pair.typeI == 'acceptable', 'acceptable', np.where(pair.typeI == 'demonstrated inflation', 'invalid', 'unclassified'))
cal = pair[(pair.split == 'calibration') & (pair.cls == 'acceptable')]
TAU_A = max(0.10, float(np.quantile(cal.A, .9))) if len(cal) else np.nan
ev = pair[pair.split == 'evaluation'].assign(flag=lambda d: d.A > TAU_A)
flags = ev.groupby(['sampler', 'world', 'cls']).flag.mean().rename('flag rate').reset_index()
sens = flags[flags.cls == 'invalid']['flag rate'].mean() if (flags.cls == 'invalid').any() else np.nan
fa = flags[flags.cls == 'acceptable']['flag rate'].max() if (flags.cls == 'acceptable').any() else np.nan
AUDIT_OK = 'inconclusive (no invalid pair)' if np.isnan(sens) else bool(sens >= .80 and fa <= .20)
print(f'τ_A = {TAU_A:.3f}; mean flag rate over invalid pairs {sens:.2f} (≥ 0.80); largest flag rate of an acceptable pair {fa:.2f} (≤ 0.20) -> audit validated: {AUDIT_OK}')
flags.pivot_table(index=['sampler', 'cls'], columns='world', values='flag rate')

audit: 13.7 min
τ_A = 0.562; mean flag rate over invalid pairs nan (≥ 0.80); largest flag rate of an acceptable pair 0.45 (≤ 0.20) -> audit validated: inconclusive (no invalid pair)


world                 W1 regime  W2 volatility  W3 long memory
sampler cls                                                   
M-a     acceptable         0.45            NaN            0.00
        unclassified        NaN           0.60             NaN
M-b     unclassified       0.90           0.00            0.05
M-c     acceptable          NaN           0.35             NaN
        unclassified       0.00            NaN            0.00
M-d     unclassified       0.90           0.00            0.00
M-e     acceptable         0.20           0.00             NaN
        unclassified        NaN            NaN            1.00
P       acceptable         0.00           0.00             NaN
        unclassified        NaN            NaN            0.00
R       unclassified       0.00           0.00            0.00

In [9]:
fam_cols = [c for c in AUD if c.startswith('share_')]
AUD.groupby('sampler')[fam_cols + ['A']].median().loc[list(SAMPLERS)]

,share_H1,share_H2,share_H3,share_H4,share_H5,share_H6,A
sampler,,,,,,,
P,0.132,0.177,0.097,0.122,0.125,0.155,0.215
R,0.177,0.201,0.097,0.178,0.062,0.153,0.208
M-a,0.253,0.542,0.097,0.207,0.250,0.162,0.556
M-b,0.339,0.514,0.090,0.367,0.188,0.217,0.514
M-c,0.135,0.243,0.090,0.138,0.250,0.151,0.271
M-d,0.132,0.181,0.097,0.242,0.125,0.162,0.250
M-e,0.488,0.438,0.104,0.105,0.125,0.122,0.507


## 6 · Decisions fixed for notebook 20 (plan §8)

In [10]:
if ELIGIBLE:
    p05 = pw.xs(.05, level='c').power.groupby(level='sampler').mean()
    ub = TAB.loc[ELIGIBLE].hi.groupby(level='sampler').max()
    CHOSEN = p05.idxmax() if len(ELIGIBLE) == 1 or abs(p05.max() - p05.min()) > .03 else ub.idxmin()
    POWER_OK = bool((pw.xs((CHOSEN, .05), level=('sampler', 'c')).power >= .80).all())
else: CHOSEN, POWER_OK = None, False
SUMMARY = dict(gap=GAP, eligible=ELIGIBLE, chosen=CHOSEN, tau_A=TAU_A, audit_validated=AUDIT_OK, power_ok=POWER_OK, misspecified=CLASS)
pd.to_pickle(SUMMARY, CACHE / 'summary.pkl')
SUMMARY

{'gap': 4,
 'eligible': [],
 'chosen': None,
 'tau_A': 0.5625,
 'audit_validated': 'inconclusive (no invalid pair)',
 'power_ok': False,
 'misspecified': {'M-a': 'unclassified',
  'M-b': 'unclassified',
  'M-c': 'unclassified',
  'M-d': 'unclassified',
  'M-e': 'unclassified'}}

## 7 · Reading

**Outcome under the declared rules (plan §4, §8): no eligible candidate.** Notebook 20 records *closed*, and NEON Q0 development ends for this phase. The run took 9 h 10 min, within the 10 h cap of plan §9, so no pairs were truncated.

**The candidates were not shown to hold level within the 0.10 tolerance.** The one-sided 95 % Clopper–Pearson interval is shown after each count.

| candidate | W0R | W0P | W1 regime | W2 volatility | W3 long memory | pooled (2000) |
|---|---|---|---|---|---|---|
| P | 32 (0.059–0.106) | 14 | 29 | 24 | 30 (0.054–0.100) | 0.065 |
| R | 29 | 20 | 34 (0.063–0.112) | 30 | 45 (0.087–0.142) | 0.079 |

- **P failed the gate on precision.** It exceeds the acceptable limit of 29 rejections by 3 in W0R and by 1 in W3. Neither candidate shows demonstrated inflation in any world.
- **The gate's own error rate.** A sampler holding exactly 0.05 would pass all five worlds with probability 0.91 at n = 400. So P's failure alone does not prove it invalid. Its pooled rate is 0.065 (binomial p = 0.004 against 0.05), so P is probably mildly anti-conservative, at roughly 0.06–0.08.
- **R is worse.** Its pooled rate is 0.079, and it is driven by the long-memory world: 45/400, lower bound 0.087.

**The misspecified samplers did not do what the design needed.**
- No misspecified sampler showed demonstrated inflation, so none is classed *invalid*. Their pooled rates are:

  | M-a (notebook 17 sampler) | M-b (φ_w = 0) | M-c (Gaussian, no scale) | M-d (independent features) | M-e (persist × 1.5) |
  |---|---|---|---|---|
  | 0.049 | 0.075 | 0.066 | 0.089 | 0.042 |

- The samplers that failed the real-image audit in notebook 17 (M-a) or simulate persistence too high (M-e) held level *better* than either candidate.
- **Reading:** in these worlds, with this statistic, the CRT's Type I error responds only weakly to sampler misspecification, between 0.04 and about 0.11. The repairs bought no measurable validity.

**This is also a weakness of the design (plan §4).**
- **The arms were too small to classify most samplers.** At n = 200, an exact-level sampler would be classed acceptable in all five worlds only 32 % of the time. A sampler at a true rate of 0.15 would be flagged as inflated only 68 % of the time.
- **Audit validation was unlikely to be possible.** It needs an invalid pair, and that required a misspecification that inflated Type I to about 0.15 or more. None of the five did.
- **These operating characteristics were not computed before the run, and should have been.**

**Held-out audit (§5): inconclusive by the declared rule.**
- The audit cannot be validated because no pair is invalid.
- Descriptively, A separates the candidates from most misspecified samplers:

  | | P | R | M-a … M-e |
  |---|---|---|---|
  | mean A | 0.21–0.23 | 0.21–0.22 | 0.15–0.72 |

- The misfit A detects did not translate into Type I inflation.
- The acceptable pair M-a × W1 has a flag rate of 0.45 at τ_A = 0.562, above the 0.20 limit.
- The audit therefore measures distributional misfit, not validity.

**What this does and does not show.**
- It does not show that CPER images carry no moisture information. Q0 was not run, and the real labels were not read.
- It does not show that the repaired samplers are invalid. They are not shown valid at the required precision. In every null world tried, the rate stays between about 0.04 and 0.11, depending on sampler and world.
- **The methods result:** for this CRT on CPER-sized data, distributional fidelity of the visual sampler (as audited) is a poor proxy for Type I validity. Certifying a sampler at ±0.05 of nominal would need on the order of 1000 datasets per world. The excess concentrates in long-memory and resampled-label worlds (pooled 0.084 and 0.070).

Both points carry over to the field pilot. A field CRT needs either a sampler fitted from much more independent data, or a design (several sites or years) in which the null can be calibrated by replication rather than by simulation.